In [ ]:
import os
from pathlib import Path
from io import StringIO
import dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from langchain_openai import AzureChatOpenAI

dotenv.load_dotenv()

In [ ]:
model: AzureChatOpenAI = None
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# System prompt now explicitly instructs incremental, idempotent updates across batches.
system_prompt = """
You are an expert ontology engineer. You will receive the domain standard as sequential page IMAGES in SMALL BATCHES.
Create an ontology in OWL/XML for a tensile strength test based on BFO (Basic Formal Ontology).
Structure it to include relevant high-level classes like specimens, equipment, test data, conditions, standards etc., and give each of them at least 4 meaningful subclasses.
The ontology should fit use in an adhesive company and use the existing onthology to expand, extend and update.
TASK:
- Extend and refine an existing OWL ontology incrementally (never restart from scratch) based on the newly provided page images.
- Preserve ALL existing IRIs, classes, properties, axioms unless a clearly superior hierarchical placement is needed; do not delete information unless it is an exact duplicate.
- Only ADD or UPDATE when NEW normative content appears in the current batch of images.
- Avoid duplicate class / property creation (match by label ignoring case & whitespace; also use synonyms to detect matches).
- Maintain stable identifiers: if a term already exists (preferred term OR any synonym), reuse / enrich it.
- Extract: Classes, ObjectProperties, DataProperties, Synonyms, Deprecated flags, Domain / Range, Hierarchies informed by numeric qualifiers.
- Carry numeric qualifiers into rdfs:label (e.g., TensileStrength_23C_50mmMin).
- If a definition starts with <...> (scoping bracket), annotate with an rdfs:comment capturing the bracketed limitation.
- Mark deprecated terms using owl:deprecated "true"^^xsd:boolean.
- Add new synonyms via skos:altLabel. Preferred via rdfs:label.
- Keep ontology valid OWL/XML serialization (no markdown fences). Root ontology element only once.
- IMPORTANT INCREMENTAL BEHAVIOR: You ALWAYS receive the CURRENT ontology text followed by the new batch of up to 10 images. Update it and return ONLY the FULL UPDATED ontology. 
- DO NOT ADD SPACES, COMMENTS OR LINE BREAKS THAT ARE NOT NEEDED TO MINIMIZE TOKEN USAGE
"""

task_prompt = """
INSTRUCTIONS FOR EACH BATCH:
1. Parse the supplied existing ontology (first content block of type text before images).
2. Analyze ONLY the newly provided images for additional concepts.
3. Merge changes.
4. Output the COMPLETE UPDATED ontology (OWL/XML). DO NOT wrap in code fences. DO NOT include explanations.
"""

In [ ]:
token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default")

model = AzureChatOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        azure_deployment=os.getenv("AZURE_OPENAI_COMPLETION_DEPLOYMENT_NAME"),
        openai_api_version=os.getenv("AZURE_OPENAI_VERSION"),
        temperature=1
    )

In [ ]:
import base64
from io import BytesIO
from PIL import Image

# --- Image Utilities ---

def convert_to_base64(pil_image: Image.Image, format_: str = "PNG") -> str:
    """Convert a PIL Image to a Base64 data string."""
    buffered = BytesIO()
    pil_image.save(buffered, format=format_)
    return base64.b64encode(buffered.getvalue()).decode("utf-8")


def load_image(image_path: str) -> Image.Image:
    return Image.open(image_path)


In [ ]:
image_path = "../../data/DIN EN ISO 472_2013"

initial_onthology_file = "initial_onthology.xml"  # Baseline ontology that is always the conceptual starting point
ontology_file_path = "ontology_iso_472.xml"       # Incrementally updated working ontology (output only)

# --- Sliding Window + Batch Processing Configuration ---
from typing import List, Iterable

BATCH_SIZE = 8     # number of images per batch
OVERLAP = 2         # number of images reused from previous batch (sliding window overlap)

all_image_files = [
    f for f in os.listdir(image_path)
    if os.path.isfile(os.path.join(image_path, f)) and f.lower().endswith((".png", ".jpg", ".jpeg"))
]
all_image_files.sort()

# all_image_files=all_image_files[0:10]

def sliding_windows(files: List[str], batch_size: int, overlap: int) -> Iterable[List[str]]:
    if batch_size <= 0:
        raise ValueError("batch_size must be > 0")
    if overlap < 0:
        raise ValueError("overlap must be >= 0")
    if overlap >= batch_size:
        raise ValueError("overlap must be < batch_size")
    start = 0
    step = batch_size - overlap
    while start < len(files):
        batch = files[start:start + batch_size]
        if not batch:
            break
        yield batch
        if start + batch_size >= len(files):
            break
        start += step


def build_message(batch_files: List[str], ontology_text: str) -> HumanMessage:
    content_blocks = [{"type": "text", "text": system_prompt + task_prompt + ontology_text}]
    for bf in batch_files:
        img_b64 = convert_to_base64(load_image(os.path.join(image_path, bf)))
        content_blocks.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{img_b64}", "detail": "low"}
        })
    return HumanMessage(content=content_blocks)


def ensure_initial_ontology(path: str):
    """Ensure the initial baseline ontology file exists; if not, create a minimal one."""
    if not os.path.exists(path):
        minimal = """<?xml version=\"1.0\"?>\n<Ontology xmlns=\"http://www.w3.org/2002/07/owl#\"\n           xml:base=\"http://example.org/iso472\"\n           xmlns:rdf=\"http://www.w3.org/1999/02/22-rdf-syntax-ns#\"\n           xmlns:xml=\"http://www.w3.org/XML/1998/namespace\"\n           xmlns:xsd=\"http://www.w3.org/2001/XMLSchema#\"\n           xmlns:rdfs=\"http://www.w3.org/2000/01/rdf-schema#\"\n           xmlns:skos=\"http://www.w3.org/2004/02/skos/core#\"\n           ontologyIRI=\"http://example.org/iso472\">\n</Ontology>"""
        with open(path, "w") as f:
            f.write(minimal)


def read_ontology(path: str) -> str:
    with open(path, "r") as f:
        return f.read()


def write_ontology(path: str, content: str):
    cleaned = content.replace("```xml", "").replace("```", "").strip()
    with open(path, "w") as f:
        f.write(cleaned)


def process_batches():
    # Always load (not copy) the baseline ontology content into memory as the starting point.
    ensure_initial_ontology(initial_onthology_file)
    current_ontology = read_ontology(initial_onthology_file)
    batch_index = 0
    for batch in sliding_windows(all_image_files, BATCH_SIZE, OVERLAP):
        print(f"Processing batch {batch_index + 1}: {batch}")
        message = build_message(batch, current_ontology)
        response = model.invoke([message])
        write_ontology(ontology_file_path, response.content)
        current_ontology = response.content  # advance in-memory ontology for next batch
        print(f"Updated ontology after batch {batch_index + 1} written.")
        batch_index += 1
    print("All batches processed.")



In [ ]:
# Guard to avoid running multiple times in the same kernel
if not globals().get("_process_batches_called", False):
    _process_batches_called = True
    process_batches()
else:
    print("process_batches() already executed in this kernel session; skipping.")